# <p style="color:#07d839; font-family:sans-serif;"> 😀 Sentiment Analysis — Análisis de Sentimiento 🥲 </p>

---

## <p style="color:#07d839;">🎯 Objetivo de la Tarea</p>

El propósito de este análisis es desentrañar la carga emocional oculta tras las letras de diferentes géneros musicales. Buscamos responder preguntas clave sobre la naturaleza sentimental de la música:

*   **Polaridad:** ¿Qué géneros son predominantemente positivos, negativos o neutros?
*   **Intensidad:** ¿Existen géneros con una carga emocional negativa o positiva mucho más marcada?
*   **Emociones específicas:** ¿Qué géneros destacan en sentimientos como la **tristeza**, **ira**, **alegría** o **amor**?
*   **Variabilidad:** ¿Es el sentimiento constante dentro de un género o existe mucha disparidad entre canciones?

---

## <p style="color:#07d839;">⚙️ Metodología del Análisis</p>

La unidad básica de estudio es **cada canción individual**. El flujo de procesamiento sigue esta estructura lógica:

> **Flujo de Trabajo:**
> `Canción` ➔ `Letra (Texto)` ➔ `Modelo de Sentimiento` ➔ `Score (Positivo/Negativo/Neutro)` | `Top Sentimientos por género`



---

In [120]:
# Librerías
import ast
import gc
import glob
import html
import os
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import torch
from matplotlib.patches import Patch
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

---
### **1. Limpieza dataset**

+ Eliminar canciones sin lyrics.

+ Eliminar canciones sin ningún género válido.

+ Quitar letras demasiado cortas, por ejemplo menos de 30 o 50 palabras.

+ Normalizar nombres de géneros: pop, Pop, POP → pop.

+ Revisar duplicados por artist + song.

+ NLP sobre lyrics para preparar análisis de sentimiento 

Limpiar la letra sin destruir información útil para el sentimiento. En análisis de sentimiento no conviene eliminar stopwords agresivamente, porque palabras como not, never, no, without cambian completamente el significado.

In [121]:
# cargar dataset
DATA_PATH = "outputs/songs_clean_by_genre_exploded_transformer.csv"

df = pd.read_csv(DATA_PATH)

print("Shape inicial:", df.shape)
display(df.head())
display(df.tail())

Shape inicial: (9189, 16)


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,artist_clean,genre_1_clean,genre_2_clean,genre_3_clean,genre,n_genres,word_count,artist_norm,song_norm
0,/moonspell/,Heartshaped Abyss,Never resist\r\nThe firewalker\r\nHe has been ...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,167,/moonspell/,heartshaped abyss
1,/moonspell/,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,197,/moonspell/,let the children cum to me...
2,/moonspell/,Memento Mori,"I am the moment, the soul\r\nThe moment that e...","gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,219,/moonspell/,memento mori
3,/moonspell/,Once It Was Ours!,Vanishing act inside the weak\r\nIn need of yo...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,174,/moonspell/,once it was ours!
4,/moonspell/,Serpent Angel,Father Satan send the Serpent\r\nPoison me wit...,"gothic metal, doom metal, dark metal",gothic metal,doom metal,dark metal,moonspell,Metal,Metal,Metal,Metal,1,152,/moonspell/,serpent angel


,artist,song,lyrics,genres,genre_1,genre_2,genre_3,artist_clean,genre_1_clean,genre_2_clean,genre_3_clean,genre,n_genres,word_count,artist_norm,song_norm
9184,Rolling Stones,Oh Baby (we Got A Good Thing Goin'),No matter how much she wants me\r\n\r\nI'm not...,"classic rock, rock, 60s",classic rock,rock,60s,Rolling Stones,Rock,Rock,Other,Other,2,201,rolling stones,oh baby (we got a good thing goin')
9185,Rolling Stones,Sister Morphine,"Here I lie in my hospital bed\r\n\r\nTell me, ...","classic rock, rock, 70s",classic rock,rock,70s,Rolling Stones,Rock,Rock,Other,Rock,2,172,rolling stones,sister morphine
9186,Rolling Stones,Sister Morphine,"Here I lie in my hospital bed\r\n\r\nTell me, ...","classic rock, rock, 70s",classic rock,rock,70s,Rolling Stones,Rock,Rock,Other,Other,2,172,rolling stones,sister morphine
9187,Rolling Stones,Through The Lonely Nights,Through the lonely nights I think of you\r\n\r...,"classic rock, rock, 60s",classic rock,rock,60s,Rolling Stones,Rock,Rock,Other,Rock,2,177,rolling stones,through the lonely nights
9188,Rolling Stones,Through The Lonely Nights,Through the lonely nights I think of you\r\n\r...,"classic rock, rock, 60s",classic rock,rock,60s,Rolling Stones,Rock,Rock,Other,Other,2,177,rolling stones,through the lonely nights


In [122]:
print("Columnas disponibles:")
display(df.columns.tolist())

Columnas disponibles:


['artist',
 'song',
 'lyrics',
 'genres',
 'genre_1',
 'genre_2',
 'genre_3',
 'artist_clean',
 'genre_1_clean',
 'genre_2_clean',
 'genre_3_clean',
 'genre',
 'n_genres',
 'word_count',
 'artist_norm',
 'song_norm']

#### **1.1 NLP**

**PREPROCESADO DE LYRICS**

In [123]:
def clean_lyrics(text):
    """
    Limpieza conservadora de letras para análisis emocional.
    No elimina stopwords ni aplica stemming/lemmatization.
    """
    
    if pd.isna(text):
        return np.nan
    
    text = str(text)
    
    # Decodificar entidades HTML
    text = html.unescape(text)
    
    # Eliminar URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    
    # Normalizar comillas raras
    text = (
        text.replace("“", '"')
            .replace("”", '"')
            .replace("‘", "'")
            .replace("’", "'")
    )
    
    # Eliminar etiquetas típicas de letras
    section_patterns = [
        r"\[.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\]",
        r"\(.*?(verse|chorus|bridge|intro|outro|hook|pre-chorus|post-chorus|interlude|refrain).*?\)"
    ]
    
    for pattern in section_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    
    # Eliminar patrones frecuentes de Genius u otras fuentes
    text = re.sub(r"\d+\s*contributors?", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"you might also like", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"embed$", " ", text, flags=re.IGNORECASE)
    
    # Normalizar saltos de línea y tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")
    
    # Espacios múltiples
    text = re.sub(r"\s+", " ", text).strip()
    
    if text == "":
        return np.nan
    
    return text

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)

def count_words(text):
    if pd.isna(text):
        return 0
    return len(re.findall(r"\b[\w']+\b", str(text)))

df["word_count"] = df["lyrics_clean"].apply(count_words)

display(df[["artist_clean", "song", "lyrics_clean", "word_count"]].head())

,artist_clean,song,lyrics_clean,word_count
0,moonspell,Heartshaped Abyss,Never resist The firewalker He has been sent T...,167
1,moonspell,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...",197
2,moonspell,Memento Mori,"I am the moment, the soul The moment that ends...",219
3,moonspell,Once It Was Ours!,Vanishing act inside the weak In need of you a...,174
4,moonspell,Serpent Angel,Father Satan send the Serpent Poison me with y...,152


En primer lugar, se realizó una limpieza del dataset original con el objetivo de preparar las letras de las canciones y los géneros musicales para el análisis emocional posterior. Durante esta fase se eliminaron canciones sin letra disponible, canciones con letras demasiado cortas y registros sin ningún género válido. Además, se calculó el número de palabras de cada lyric para poder filtrar aquellas letras que no contenían suficiente información textual para el análisis.

La limpieza aplicada sobre las letras fue conservadora, ya que el objetivo posterior era utilizar un modelo Deep Learning/Transformer de clasificación emocional. Por ello, el preprocesado se centró únicamente en eliminar ruido técnico o elementos que no formaban parte real de la letra. En concreto, se eliminaron URLs, etiquetas estructurales de canciones como [Verse], [Chorus] o [Bridge], patrones procedentes del scraping y espacios duplicados. También se normalizaron comillas y entidades HTML para mejorar la consistencia del texto.

No se aplicaron técnicas agresivas de preprocesado textual. En concreto, no se eliminaron stopwords, no se aplicó stemming, no se aplicó lematización, no se eliminaron signos de puntuación de forma agresiva, no se convirtió todo el texto a minúsculas, no se eliminaron negaciones como "not", "no" o "never", y no se realizó una tokenización manual orientada al entrenamiento de un modelo clásico.

Esta decisión se debe a que los modelos Deep Learning/Transformer utilizan el contexto completo de la oración para interpretar el significado del texto. Un preprocesado excesivamente agresivo, como eliminar palabras comunes, modificar la forma original de las palabras o suprimir signos de puntuación, puede destruir la estructura gramatical y eliminar matices emocionales importantes. En tareas de análisis emocional, palabras aparentemente simples, negaciones y signos de puntuación pueden cambiar de forma relevante el sentido de una frase. Por este motivo, se decidió conservar el texto lo más natural posible y limitar la limpieza a la eliminación de etiquetas, metadatos y ruido técnico.

Además del preprocesado de letras, se normalizaron las etiquetas de género musical presentes en las columnas genre_1, genre_2 y genre_3. Para ello, se convirtieron los géneros a minúsculas, se eliminaron espacios innecesarios y se unificaron algunas variantes frecuentes. Dado que una misma canción puede pertenecer a varios géneros, se generó también un segundo dataset en formato expandido, donde cada fila representa una combinación canción-género. Este formato permite realizar posteriormente un análisis más completo de la distribución emocional de las letras para cada género musical.

Archivos generados:

| Archivo                                      | Nivel                       | Uso principal                   |
| -------------------------------------------- | --------------------------- | ------------------------------- |
| `dataset_fusionado.csv`                    | Una fila por canción        | Visualización compacta     |
| `songs_clean_by_genre_exploded.csv`          | Una fila por canción-género | Análisis completo por género    |



---
### **2. Clasificación de canciones en sentimiento.**

`Modelo 1`: modelo de sentimiento con 3 clases. Un modelo como cardiffnlp/twitter-roberta-base-sentiment-latest realiza la tarea más fundamental (análisis de polaridad), clasificando el texto en tres categorías básicas: positivo, negativo o neutral.

`Modelo 2`: modelo de emociones con 7 clases. Un modelo como j-hartmann/emotion-english-distilroberta-base sube un nivel en complejidad al clasificar el texto en emociones básicas de Ekman (etiqueta única o single-label): anger (ira), disgust (asco), fear (miedo), joy (alegría), neutral, sadness (tristeza) y surprise (sorpresa).

`Modelo 3`: modelo GoEmotions con 28 emociones. Un modelo como SamLowe/roberta-base-go_emotions representa la mayor complejidad y granularidad. Está entrenado sobre el dataset GoEmotions y funciona como clasificación multi-label; es decir, entiende que los sentimientos humanos son complejos y permite que una misma letra active varias emociones simultáneas (como amor, optimismo y gratitud al mismo tiempo).

In [124]:
#!pip install transformers torch tqdm -q

In [125]:
# cargar datoset limpio por canción

df_songs = df.copy()

print("Shape:", df_songs.shape)
display(df_songs[["artist_clean", "song", "lyrics_clean", "word_count"]].head())

df_songs = df_songs[df_songs["lyrics_clean"].notna()].copy()
df_songs = df_songs.reset_index(drop=True)

print("Canciones válidas:", len(df_songs))

Shape: (9189, 17)


,artist_clean,song,lyrics_clean,word_count
0,moonspell,Heartshaped Abyss,Never resist The firewalker He has been sent T...,167
1,moonspell,Let The Children Cum To Me...,"Hey you little Jesus' bride, why have you smil...",197
2,moonspell,Memento Mori,"I am the moment, the soul The moment that ends...",219
3,moonspell,Once It Was Ours!,Vanishing act inside the weak In need of you a...,174
4,moonspell,Serpent Angel,Father Satan send the Serpent Poison me with y...,152


Canciones válidas: 9189


In [126]:
# CONFIGURACIÓN GENERAL MODELOS que vamos a usar para análisis

TEXT_COL = "lyrics_clean"

MODEL_CONFIGS = [
    {
        "model_key": "cardiff",
        "model_name": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "task_type": "single_label",
        "output_col": "cardiff_sentiment",
        "score_col": "cardiff_sentiment_score",
        "top_k": 1,
        "exclude_neutral": True
    },
    {
        "model_key": "jhartmann",
        "model_name": "j-hartmann/emotion-english-distilroberta-base",
        "task_type": "single_label",
        "output_col": "jhartmann_emotion",
        "score_col": "jhartmann_emotion_score",
        "top_k": 1,
        "exclude_neutral": True
    },
    {
        "model_key": "goemotions",
        "model_name": "SamLowe/roberta-base-go_emotions",
        "task_type": "multi_label",
        "output_col": "goemotions_top3",
        "score_col": "goemotions_top3_scores",
        "top_k": 3,
        "exclude_neutral": True,
        "exclude_approval": True
    }
]

Excluimos la etiqueta "neutral" en todos los modelos porque la música es intrínsecamente expresiva; ignorarla evita que el peso emocional de la canción se diluya con frases puramente descriptivas. Por su parte, descartamos "approval" en GoEmotions porque palabras como "yeah" o "ok", que en las canciones actúan como simples muletillas rítmicas, generan falsos positivos al ser interpretadas erróneamente como actos de aprobación. Con este filtro contextual, obligamos a los modelos a capturar exclusivamente el verdadero núcleo sentimental de la obra.

In [127]:
# DISPOSITIVO DE PROCESAMIENTO 
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


device = get_device()


In [128]:
# FUNCIONES GENERALES - DIVIDIR LETRAS LARGAS EN CHUNKS
MAX_TOKENS = 500
STRIDE = 250


def split_text_into_chunks(text, tokenizer, max_tokens=MAX_TOKENS, stride=STRIDE):
    """
    Divide una letra larga en fragmentos de tokens.
    Esto evita que el modelo analice solo el principio de la canción.
    """

    if pd.isna(text) or str(text).strip() == "":
        return []

    text = str(text)

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) == 0:
        return []

    chunks = []
    step = max_tokens - stride

    for start in range(0, len(token_ids), step):
        end = start + max_tokens
        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        if chunk_text.strip():
            chunks.append(chunk_text.strip())

        if end >= len(token_ids):
            break

    return chunks



# NORMALIZAR ETIQUETAS DE LOS MODELOS

def normalize_label(label, model_key=None):
    """
    Normaliza etiquetas devueltas por los modelos.
    """

    label = str(label).lower().strip()

    # Por seguridad, si algún modelo usa LABEL_0 / LABEL_1
    if model_key == "cardiff":
        label_mapping = {
            "label_0": "negative",
            "label_1": "positive",
            "0": "negative",
            "1": "positive"
        }
        label = label_mapping.get(label, label)

    return label



# OBTENER LOS SCORES DE UNA CANCIÓN


def predict_text_scores(
    text,
    tokenizer,
    model,
    labels,
    device,
    task_type,
    batch_size=8
):
    """
    Aplica un modelo a una canción completa.
    1. Divide la letra en chunks.
    2. Predice scores por chunk.
    3. Promedia los scores para obtener una predicción global por canción.
    """

    chunks = split_text_into_chunks(text, tokenizer)

    if len(chunks) == 0:
        return {
            "scores": {label: np.nan for label in labels},
            "n_chunks": 0
        }

    all_scores = []

    for i in range(0, len(chunks), batch_size):
        batch_chunks = chunks[i:i + batch_size]

        inputs = tokenizer(
            batch_chunks,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            if task_type == "single_label":
                probs = torch.softmax(outputs.logits, dim=1)

            elif task_type == "multi_label":
                probs = torch.sigmoid(outputs.logits)

            else:
                raise ValueError(f"task_type no reconocido: {task_type}")

        all_scores.append(probs.detach().cpu().numpy())

    all_scores = np.vstack(all_scores)
    mean_scores = all_scores.mean(axis=0)

    score_dict = {
        label: float(score)
        for label, score in zip(labels, mean_scores)
    }

    return {
        "scores": score_dict,
        "n_chunks": len(chunks)
    }




# EXTRAER LA PREDICCIÓN FINAL


def extract_prediction_from_scores(
    scores,
    model_key,
    top_k=1,
    exclude_neutral=False,
    exclude_approval=False
):
    """
    Extrae la predicción final a partir de los scores medios.
    
    Para car y j-hartmann:
    - devuelve top 1.
    
    Para GoEmotions:
    - devuelve top K emociones.
    """

    valid_scores = {
        emotion: score
        for emotion, score in scores.items()
        if not pd.isna(score)
    }

    if exclude_neutral:
        valid_scores = {
            emotion: score
            for emotion, score in valid_scores.items()
            if emotion != "neutral"
        }

    if exclude_approval:
        valid_scores = {
            emotion: score
            for emotion, score in valid_scores.items()
            if emotion != "approval"
        }

    if len(valid_scores) == 0:
        if top_k == 1:
            return None, np.nan
        else:
            return "", ""

    sorted_scores = sorted(
        valid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    if top_k == 1:
        top_label, top_score = sorted_scores[0]
        return top_label, top_score

    top_items = sorted_scores[:top_k]

    top_labels = [label for label, score in top_items]
    top_scores = [round(score, 4) for label, score in top_items]

    return ", ".join(top_labels), ", ".join(map(str, top_scores))





**`split_text_into_chunks`**: Divide la canción en fragmentos superpuestos de tokens para superar el límite de 512 de los Transformers. Usa una ventana deslizante ("stride") para que los bloques se solapen, asegurando que no se pierda el contexto gramatical ni el sentido emocional en los puntos de corte de la letra.

**`normalize_label`**: Limpia y estandariza las etiquetas devueltas por los modelos (minúsculas y sin espacios). Incluye una regla específica para el modelo CardiffNLP: traduce formatos crudos o numéricos ambiguos como "label_0" o "1" a sus equivalentes "negative" o "positive", unificando el formato de salida.

**`predict_text_scores`**: Es el motor de inferencia. Envía los fragmentos de la canción al modelo por lotes. Aplica funciones matemáticas (Softmax o Sigmoide según el tipo de tarea) para calcular probabilidades de 0 a 1. Finalmente, promedia las puntuaciones de todos los fragmentos para dar una evaluación global.

**`extract_prediction_from_scores`**: Filtra las probabilidades promediadas eliminando emociones no deseadas ("neutral" o "approval"). Luego, ordena los resultados por puntuación y devuelve la emoción ganadora (Top-1) o una lista de las principales separadas por comas (Top-K, para GoEmotions), listas para guardar en tu dataset.

In [129]:
# FUNCIÓN PARA EJECUTAR CUALQUIER MODELO
# POSTERIORMENTE HAREMOS QUE SE APLIQUE A LOS 3 MODELOS QUE QUEREMOS COMPARAR


def run_model_on_df(
    df,
    model_config,
    text_col=TEXT_COL,
    device=device,
    batch_size=8,
    save_all_scores=False
):
    """
    Aplica un modelo Transformer a todas las canciones de df.
    
    Devuelve un DataFrame con:
    - predicción principal
    - score principal
    - número de chunks
    - opcionalmente, scores de todas las etiquetas
    """

    model_key = model_config["model_key"]
    model_name = model_config["model_name"]
    task_type = model_config["task_type"]
    output_col = model_config["output_col"]
    score_col = model_config["score_col"]
    top_k = model_config["top_k"]
    exclude_neutral = model_config["exclude_neutral"]

    print("=" * 90)
    print(f"Cargando modelo: {model_name}")
    print("=" * 90)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    model.to(device)
    model.eval()

    id2label = model.config.id2label

    labels = [
        normalize_label(id2label[i], model_key=model_key)
        for i in range(len(id2label))
    ]

    print("Etiquetas del modelo:")
    print(labels)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Aplicando {model_key}"):

        text = row[text_col]

        prediction_result = predict_text_scores(
            text=text,
            tokenizer=tokenizer,
            model=model,
            labels=labels,
            device=device,
            task_type=task_type,
            batch_size=batch_size
        )

        scores = prediction_result["scores"]
        n_chunks = prediction_result["n_chunks"]

        pred_label, pred_score = extract_prediction_from_scores(
            scores=scores,
            model_key=model_key,
            top_k=top_k,
            exclude_neutral=exclude_neutral
        )

        result = {
            output_col: pred_label,
            score_col: pred_score,
            f"{model_key}_n_chunks": n_chunks
        }

        if save_all_scores:
            for label, score in scores.items():
                result[f"{model_key}_{label}"] = score

        results.append(result)

    df_results = pd.DataFrame(results)

    # Liberar memoria
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return df_results

**`run_model_on_df`**: Orquesta el flujo completo. Carga el modelo en la GPU y procesa el dataset iterando cada canción con las funciones previas para predecir su emoción. Tras guardar los resultados en un DataFrame, libera exhaustivamente la memoria de la tarjeta gráfica para evitar cuelgues.

In [130]:
# APLICAR A LOS 3 MODELOS

df_model_comparison = df_songs.copy().reset_index(drop=True)

# Aseguramos que solo analizamos canciones con letra limpia válida
df_model_comparison = df_model_comparison[
    df_model_comparison[TEXT_COL].notna()
].copy()

df_model_comparison = df_model_comparison.reset_index(drop=True)

print("Canciones a analizar:", len(df_model_comparison))

Canciones a analizar: 9189


In [131]:
# EJECUTAR

for config in MODEL_CONFIGS:

    model_results = run_model_on_df(
        df=df_model_comparison,
        model_config=config,
        text_col=TEXT_COL,
        device=device,
        batch_size=8,
        save_all_scores=False
    )

    df_model_comparison = pd.concat(
        [
            df_model_comparison.reset_index(drop=True),
            model_results.reset_index(drop=True)
        ],
        axis=1
    )

    display(
        df_model_comparison[
            [
                "artist",
                "song",
                config["output_col"],
                config["score_col"],
                f"{config['model_key']}_n_chunks"
            ]
        ].head()
    )

Cargando modelo: cardiffnlp/twitter-roberta-base-sentiment-latest


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40672.28it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Etiquetas del modelo:
['negative', 'neutral', 'positive']


Aplicando cardiff: 100%|██████████| 9189/9189 [06:40<00:00, 22.95it/s]


,artist,song,cardiff_sentiment,cardiff_sentiment_score,cardiff_n_chunks
0,/moonspell/,Heartshaped Abyss,positive,0.277471,1
1,/moonspell/,Let The Children Cum To Me...,positive,0.773513,1
2,/moonspell/,Memento Mori,positive,0.201928,1
3,/moonspell/,Once It Was Ours!,positive,0.402567,1
4,/moonspell/,Serpent Angel,negative,0.450096,1


...

Cargando modelo: SamLowe/roberta-base-go_emotions


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6565.19it/s]


Etiquetas del modelo:
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


Aplicando goemotions: 100%|██████████| 9189/9189 [06:39<00:00, 22.98it/s]


,artist,song,goemotions_top3,goemotions_top3_scores,goemotions_n_chunks
0,/moonspell/,Heartshaped Abyss,"fear, anger, annoyance","0.5027, 0.0689, 0.0458",1
1,/moonspell/,Let The Children Cum To Me...,"love, curiosity, approval","0.5677, 0.1897, 0.0838",1
2,/moonspell/,Memento Mori,"realization, approval, disappointment","0.0716, 0.0377, 0.0109",1
3,/moonspell/,Once It Was Ours!,"sadness, realization, disappointment","0.1914, 0.1131, 0.0854",1
4,/moonspell/,Serpent Angel,"approval, excitement, love","0.0193, 0.0176, 0.0168",1


In [132]:
display(
    df_model_comparison[
        [
            "artist",
            "song",
            "genre",
            "cardiff_sentiment",
            
            "jhartmann_emotion",
            
            "goemotions_top3",
            
        ]
    ].tail()
)

,artist,song,genre,cardiff_sentiment,jhartmann_emotion,goemotions_top3
9184,Rolling Stones,Oh Baby (we Got A Good Thing Goin'),Other,positive,fear,"caring, approval, love"
9185,Rolling Stones,Sister Morphine,Rock,negative,fear,"disappointment, sadness, curiosity"
9186,Rolling Stones,Sister Morphine,Other,negative,fear,"disappointment, sadness, curiosity"
9187,Rolling Stones,Through The Lonely Nights,Rock,negative,sadness,"sadness, disappointment, curiosity"
9188,Rolling Stones,Through The Lonely Nights,Other,negative,sadness,"sadness, disappointment, curiosity"


---

### **3. Agregación por género.**

Después de calcular el sentimiento por canción, se agrupa por género.


In [133]:
print("Columnas disponibles:")
display(df_model_comparison.columns.tolist())

Columnas disponibles:


['artist',
 'song',
 'lyrics',
 'genres',
 'genre_1',
 'genre_2',
 'genre_3',
 'artist_clean',
 'genre_1_clean',
 'genre_2_clean',
 'genre_3_clean',
 'genre',
 'n_genres',
 'word_count',
 'artist_norm',
 'song_norm',
 'lyrics_clean',
 'cardiff_sentiment',
 'cardiff_sentiment_score',
 'cardiff_n_chunks',
 'jhartmann_emotion',
 'jhartmann_emotion_score',
 'jhartmann_n_chunks',
 'goemotions_top3',
 'goemotions_top3_scores',
 'goemotions_n_chunks']

#### **3.1. Unir resultados de modelos con dataset expandido por género**

In [134]:
# PRIMERO UNIMOS LOS RESULTADOS DE LOS 3 MODELOS EN UN SOLO DATAFRAME EXPANDIDO POR GÉNERO
#df_model_comparison + outputs/songs_clean_by_genre_exploded_filtered.csv

df_genres_filtered = pd.read_csv("outputs/songs_clean_by_genre_exploded_transformer.csv")

print("df_model_comparison:", df_model_comparison.shape)
print("df_genres_filtered:", df_genres_filtered.shape)




# unimos las columnas que nos interesan del model_comparison con el df_genres_filtered 
# para tener un df completo con género + resultados de los modelos

model_cols = [
    "artist_norm",
    "song_norm",
    "cardiff_sentiment",
    "cardiff_sentiment_score",
    "jhartmann_emotion",
    "jhartmann_emotion_score",
    "goemotions_top3",
    "goemotions_top3_scores"
]

model_cols = [col for col in model_cols if col in df_model_comparison.columns]

df_predictions = df_model_comparison[model_cols].copy()

print("Columnas seleccionadas:", df_predictions.columns.tolist())





# HACEMOS MERGE DE LOS MODELOS CON EL DF DE GÉNEROS 
# PARA TENER UN DF COMPLETO CON ARTISTA, CANCIÓN, GÉNERO Y RESULTADOS DE LOS MODELOS

df_song_genre_models = df_genres_filtered.merge(
    df_predictions,
    on=["artist_norm", "song_norm"],
    how="left"
)

print("Shape tras merge:", df_song_genre_models.shape)



df_model_comparison: (9189, 26)
df_genres_filtered: (9189, 16)
Columnas seleccionadas: ['artist_norm', 'song_norm', 'cardiff_sentiment', 'cardiff_sentiment_score', 'jhartmann_emotion', 'jhartmann_emotion_score', 'goemotions_top3', 'goemotions_top3_scores']
Shape tras merge: (20291, 22)


In [135]:
df_genres_filtered["genre"].value_counts()

genre
Other                    3162
Metal                     979
Rock                      936
Hip Hop / Rap             790
Pop                       766
Folk / Traditional        762
Electronic / Dance        643
Country                   616
R&B / Soul / Funk         133
Latin                     125
Reggae / Caribbean         77
Blues                      74
Jazz                       51
Experimental               31
Punk / Emo / Hardcore      24
Classical / Art Music      20
Name: count, dtype: int64

Este bloque integra las predicciones de los tres modelos con la información de los géneros musicales. Primero, selecciona estrictamente las columnas de resultados (Cardiff, j-hartmann y GoEmotions). Luego, usa el artista y la canción como claves para realizar un cruce (`merge`) con el dataset de géneros, creando una base de datos maestra lista para analizar qué emociones predominan en cada estilo musical.

#### **3.2. Crear score numérico de sentimiento**

Crear un score numérico continuo basado en la polaridad del sentimiento nos permite interpretar y graficar fácilmente los resultados (creando un eje que va de -1 a +1).

Usaremos la siguiente lógica matemática:

| Etiqueta | Lógica | Ejemplo de Salida | Score Final |
| :--- | :--- | :--- | :--- |
| **Positive** | Score positivo | Positive (0.91) | `+0.91` |
| **Negative** | Score negativo | Negative (0.88) | `-0.88` |


In [136]:

def sentiment_to_signed_score(row, sentiment_col, score_col):
    label = row[sentiment_col]
    score = row[score_col]

    if pd.isna(label) or pd.isna(score):
        return np.nan

    label = str(label).lower()

    if label == "positive":
        return float(score)

    elif label == "negative":
        return -float(score)

    elif label == "neutral":
        return 0.0

    else:
        return np.nan


df_song_genre_models["sentiment_signed_score"] = df_song_genre_models.apply(
    lambda row: sentiment_to_signed_score(row, "cardiff_sentiment", "cardiff_sentiment_score"),
    axis=1
)

display(
    df_song_genre_models[
        [
            "artist",
            "song",
            "genre",
            "cardiff_sentiment",
            "cardiff_sentiment_score",
            "sentiment_signed_score"
        ]
    ].head()
)


,artist,song,genre,cardiff_sentiment,cardiff_sentiment_score,sentiment_signed_score
0,/moonspell/,Heartshaped Abyss,Metal,positive,0.277471,0.277471
1,/moonspell/,Let The Children Cum To Me...,Metal,positive,0.773513,0.773513
2,/moonspell/,Memento Mori,Metal,positive,0.201928,0.201928
3,/moonspell/,Once It Was Ours!,Metal,positive,0.402567,0.402567
4,/moonspell/,Serpent Angel,Metal,negative,0.450096,-0.450096


#### **3.3. Agregación Sentimiento por género**

Consiste en agrupar las canciones por su estilo musical y promediar los resultados obtenidos por los modelos. Esta vista macro nos permite identificar tendencias generales y descubrir qué emociones o sentimientos predominan globalmente en cada género.

##### **3.3.1. Cardiff**

In [137]:
# ============================================================
# Métricas numéricas por género
# ============================================================

genre_sentiment_summary = (
    df_song_genre_models
    .groupby("genre")
    .agg(
        n_canciones=("song_norm", "count"),
        sentimiento_medio=("sentiment_signed_score", "mean"),
        sentimiento_mediano=("sentiment_signed_score", "median"),
        desviacion_tipica=("sentiment_signed_score", "std")
    )
    .reset_index()
)

# Clase basada en el signo del sentimiento medio
genre_sentiment_summary["sentimiento_medio_clase"] = np.select(
    [
        genre_sentiment_summary["sentimiento_medio"] > 0,
        genre_sentiment_summary["sentimiento_medio"] < 0
    ],
    [
        "positive",
        "negative"
    ],
    default="neutral"
)

# ============================================================
# Porcentajes de positive / negative / neutral por género
# ============================================================

sentiment_counts = pd.crosstab(
    df_song_genre_models["genre"],
    df_song_genre_models["cardiff_sentiment"],
    normalize="index"
) * 100

sentiment_counts = sentiment_counts.round(2)

# Asegurar que existen siempre las tres columnas
for sentiment in ["negative", "neutral", "positive"]:
    if sentiment not in sentiment_counts.columns:
        sentiment_counts[sentiment] = 0.0

# Reordenar columnas
sentiment_counts = sentiment_counts[["negative", "neutral", "positive"]]

# Sentimiento más frecuente por número de canciones
sentiment_counts["sentimiento_mayoritario"] = sentiment_counts.idxmax(axis=1)

# Renombrar columnas de porcentaje
sentiment_counts = (
    sentiment_counts
    .rename(columns={
        "negative": "pct_negative",
        "neutral": "pct_neutral",
        "positive": "pct_positive"
    })
    .reset_index()
)

# ============================================================
# Tabla final
# ============================================================

genre_sentiment_final = genre_sentiment_summary.merge(
    sentiment_counts,
    on="genre",
    how="left"
)

genre_sentiment_final = (
    genre_sentiment_final
    .sort_values("genre", ascending=True)
    .reset_index(drop=True)
)

genre_sentiment_final.insert(
    0,
    "genre_id",
    range(len(genre_sentiment_final))
)

cols_order = [
    "genre",
    "sentimiento_medio_clase",
    "sentimiento_mayoritario",
    "n_canciones",
    "sentimiento_medio",
    "sentimiento_mediano",
    "desviacion_tipica",
    "pct_negative",
    "pct_neutral",
    "pct_positive"
]

genre_sentiment_final = genre_sentiment_final[cols_order]

display(genre_sentiment_final)

,genre,sentimiento_medio_clase,sentimiento_mayoritario,n_canciones,sentimiento_medio,sentimiento_mediano,desviacion_tipica,pct_negative,pct_neutral,pct_positive
0,Blues,positive,positive,193,0.171792,0.374065,0.630664,44.04,0.0,55.96
1,Classical / Art Music,positive,positive,43,0.059039,0.172233,0.488857,44.19,0.0,55.81
2,Country,positive,positive,1448,0.097818,0.253954,0.572956,43.99,0.0,56.01
3,Electronic / Dance,positive,positive,1590,0.101759,0.266043,0.581753,43.08,0.0,56.92
4,Experimental,positive,positive,82,0.014440,0.122460,0.495038,47.56,0.0,52.44
5,Folk / Traditional,positive,positive,1898,0.057097,0.207892,0.602871,46.42,0.0,53.58
6,Hip Hop / Rap,negative,negative,1460,-0.001086,-0.206696,0.509834,51.10,0.0,48.90
7,Jazz,positive,positive,123,0.153683,0.329289,0.550074,42.28,0.0,57.72
8,Latin,negative,negative,343,-0.145772,-0.387587,0.592054,64.43,0.0,35.57
9,Metal,negative,negative,1500,-0.267645,-0.446966,0.549529,70.53,0.0,29.47


El `sentimiento_mayoritario` indica qué etiqueta aparece más veces dentro de un género, sin tener en cuenta la intensidad del modelo. En cambio, el `sentimiento_medio` calcula el balance promedio usando el score firmado: positivo suma, negativo resta y neutral vale 0. Por eso puede ocurrir que un género tenga más canciones negativas, pero un sentimiento medio positivo si las canciones positivas tienen scores más altos que las negativas.


#### **3.3.2. J-hartmann**

In [138]:

# ============================================================
# Porcentajes de emociones j-hartmann por género
# ============================================================

jhartmann_emotion_pct = pd.crosstab(
    df_song_genre_models["genre"],
    df_song_genre_models["jhartmann_emotion"],
    normalize="index"
) * 100

jhartmann_emotion_pct = jhartmann_emotion_pct.round(2)

# Asegurar que existen siempre las 7 columnas del modelo
jhartmann_emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
]

for emotion in jhartmann_emotions:
    if emotion not in jhartmann_emotion_pct.columns:
        jhartmann_emotion_pct[emotion] = 0.0

# Reordenar columnas
jhartmann_emotion_pct = jhartmann_emotion_pct[jhartmann_emotions]

# Emoción predominante por género
jhartmann_emotion_pct["jhartmann_emotion_predominante"] = (
    jhartmann_emotion_pct.idxmax(axis=1)
)

# Renombrar columnas de porcentaje
jhartmann_emotion_pct = (
    jhartmann_emotion_pct
    .rename(columns={
        "anger": "pct_jhartmann_anger",
        "disgust": "pct_jhartmann_disgust",
        "fear": "pct_jhartmann_fear",
        "joy": "pct_jhartmann_joy",
        "neutral": "pct_jhartmann_neutral",
        "sadness": "pct_jhartmann_sadness",
        "surprise": "pct_jhartmann_surprise"
    })
    .reset_index()
)

# ============================================================
# Número de canciones por género
# ============================================================

jhartmann_counts = (
    df_song_genre_models
    .groupby("genre")
    .agg(n_canciones=("song_norm", "count"))
    .reset_index()
)

# ============================================================
# Tabla final
# ============================================================

genre_jhartmann_final = jhartmann_counts.merge(
    jhartmann_emotion_pct,
    on="genre",
    how="left"
)

# Ordenar géneros alfabéticamente y crear índice 0, 1, 2...
genre_jhartmann_final = (
    genre_jhartmann_final
    .sort_values("n_canciones", ascending=False)
    .reset_index(drop=True)
)

genre_jhartmann_final.insert(
    0,
    "genre_id",
    range(len(genre_jhartmann_final))
)

# Orden final de columnas
cols_order = [
    "genre",
    "jhartmann_emotion_predominante",
    "n_canciones",
    "pct_jhartmann_anger",
    "pct_jhartmann_disgust",
    "pct_jhartmann_fear",
    "pct_jhartmann_joy",
    "pct_jhartmann_neutral",
    "pct_jhartmann_sadness",
    "pct_jhartmann_surprise"
]

genre_jhartmann_final = genre_jhartmann_final[cols_order]

display(genre_jhartmann_final)



,genre,jhartmann_emotion_predominante,n_canciones,pct_jhartmann_anger,pct_jhartmann_disgust,pct_jhartmann_fear,pct_jhartmann_joy,pct_jhartmann_neutral,pct_jhartmann_sadness,pct_jhartmann_surprise
0,Other,sadness,6955,16.33,2.93,20.76,14.78,0.0,33.14,12.05
1,Rock,sadness,2030,15.67,4.33,21.18,11.33,0.0,36.35,11.13
2,Pop,sadness,2004,11.78,1.95,22.80,16.02,0.0,36.38,11.08
3,Folk / Traditional,sadness,1898,12.49,3.11,18.49,16.12,0.0,38.99,10.80
4,Electronic / Dance,sadness,1590,15.16,2.89,22.08,16.16,0.0,32.14,11.57
5,Metal,anger,1500,30.93,3.60,25.20,6.07,0.0,28.60,5.60
6,Hip Hop / Rap,anger,1460,40.62,3.49,14.93,10.07,0.0,15.82,15.07
7,Country,sadness,1448,10.98,3.25,17.54,15.75,0.0,38.19,14.30
8,R&B / Soul / Funk,sadness,347,14.99,2.88,15.85,23.05,0.0,27.67,15.56
9,Latin,sadness,343,19.53,3.21,21.57,9.33,0.0,37.61,8.75


#### **3.3.3. GoEmotions**

In [139]:
# ============================================================
# GoEmotions: resumen por género + tabla compacta
# ============================================================

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ============================================================
# 1. Preparar GoEmotions Top 3 en formato largo
# ============================================================

df_go_top3 = df_song_genre_models[
    ["artist", "song", "artist_norm", "song_norm", "genre", "goemotions_top3"]
].copy()

df_go_top3["goemotion"] = (
    df_go_top3["goemotions_top3"]
    .fillna("")
    .astype(str)
    .str.split(", ")
)

df_go_top3 = df_go_top3.explode("goemotion").reset_index(drop=True)

df_go_top3["goemotion"] = df_go_top3["goemotion"].astype(str).str.strip()

df_go_top3 = df_go_top3[
    df_go_top3["goemotion"].notna() &
    (df_go_top3["goemotion"] != "") &
    (df_go_top3["goemotion"].str.lower() != "nan")
].copy()

# ============================================================
# 2. Porcentajes de emociones GoEmotions por género
# ============================================================

goemotions = [
    "admiration",
    "amusement",
    "anger",
    "annoyance",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise",
    "neutral"
]

goemotion_counts_by_genre = pd.crosstab(
    df_go_top3["genre"],
    df_go_top3["goemotion"],
    normalize="index"
) * 100

goemotion_counts_by_genre = goemotion_counts_by_genre.round(2)

for emotion in goemotions:
    if emotion not in goemotion_counts_by_genre.columns:
        goemotion_counts_by_genre[emotion] = 0.0

goemotion_counts_by_genre = goemotion_counts_by_genre[goemotions]

goemotion_counts_by_genre["goemotions_emotion_predominante"] = (
    goemotion_counts_by_genre.idxmax(axis=1)
)

goemotion_counts_by_genre = (
    goemotion_counts_by_genre
    .rename(columns={emotion: f"pct_go_{emotion}" for emotion in goemotions})
    .reset_index()
)

# ============================================================
# 3. Tabla final completa
# ============================================================

goemotions_counts = (
    df_song_genre_models
    .groupby("genre")
    .agg(n_canciones=("song_norm", "count"))
    .reset_index()
)

genre_goemotions_final = goemotions_counts.merge(
    goemotion_counts_by_genre,
    on="genre",
    how="left"
)

genre_goemotions_final = (
    genre_goemotions_final
    .sort_values("n_canciones", ascending=False)
    .reset_index(drop=True)
)

cols_order = [
    "genre",
    "goemotions_emotion_predominante",
    "n_canciones"
] + [f"pct_go_{emotion}" for emotion in goemotions]

genre_goemotions_final = genre_goemotions_final[cols_order]

display(genre_goemotions_final)

# ============================================================
# 4. Tabla compacta: género, emoción predominante y top 3
# ============================================================

def get_top3_emotions_text(emotions):
    top3 = emotions.value_counts().head(3).index.tolist()
    return ", ".join(top3)


def get_top3_emotions_with_pct(emotions):
    counts = emotions.value_counts().head(3)
    total = len(emotions)

    return ", ".join([
        f"{emotion} ({round(count / total * 100, 2)}%)"
        for emotion, count in counts.items()
    ])


goemotions_top3_by_genre = (
    df_go_top3
    .groupby("genre")["goemotion"]
    .agg(
        top3_goemotions_mas_comunes=get_top3_emotions_text,
        top3_goemotions_mas_comunes_pct=get_top3_emotions_with_pct
    )
    .reset_index()
)

genre_goemotions_compact = (
    genre_goemotions_final[
        ["genre", "goemotions_emotion_predominante", "n_canciones"]
    ]
    .merge(
        goemotions_top3_by_genre,
        on="genre",
        how="left"
    )
    .sort_values("n_canciones", ascending=False)
    .reset_index(drop=True)
)

genre_goemotions_compact = genre_goemotions_compact[
    [
        "genre",
        "goemotions_emotion_predominante",
        "top3_goemotions_mas_comunes",
        "top3_goemotions_mas_comunes_pct"
    ]
]

display(genre_goemotions_compact)


,genre,goemotions_emotion_predominante,n_canciones,pct_go_admiration,pct_go_amusement,pct_go_anger,pct_go_annoyance,pct_go_caring,pct_go_confusion,pct_go_curiosity,...,pct_go_love,pct_go_nervousness,pct_go_optimism,pct_go_pride,pct_go_realization,pct_go_relief,pct_go_remorse,pct_go_sadness,pct_go_surprise,pct_go_neutral
0,Other,love,6955,3.26,1.55,2.74,9.22,4.07,3.37,5.19,...,11.51,0.62,3.34,0.01,4.25,0.08,0.46,8.83,0.62,0.0
1,Rock,disappointment,2030,3.17,1.08,2.22,9.26,3.88,4.38,5.67,...,8.92,0.54,3.96,0.07,5.35,0.13,0.62,9.93,0.62,0.0
2,Pop,love,2004,3.33,1.31,1.53,6.59,4.29,3.41,4.86,...,15.07,0.63,2.76,0.00,3.36,0.07,0.58,9.63,0.77,0.0
3,Folk / Traditional,love,1898,3.37,1.02,1.40,7.39,4.18,3.53,4.23,...,11.47,0.77,3.27,0.00,5.29,0.19,0.54,10.63,0.81,0.0
4,Electronic / Dance,love,1590,2.96,1.19,2.70,8.53,4.68,4.15,5.79,...,12.60,0.63,2.77,0.00,4.13,0.00,0.48,8.57,0.61,0.0
5,Metal,annoyance,1500,2.22,1.42,7.67,15.11,3.22,3.31,4.49,...,3.78,0.78,3.11,0.07,5.93,0.02,0.22,10.76,0.58,0.0
6,Hip Hop / Rap,annoyance,1460,3.29,4.47,11.92,20.11,2.03,1.92,4.29,...,8.15,0.23,2.28,0.00,1.85,0.00,0.32,4.02,0.30,0.0
7,Country,love,1448,3.80,1.57,1.59,7.57,3.41,2.28,3.22,...,13.33,0.71,3.59,0.02,4.93,0.18,0.30,9.78,0.41,0.0
8,R&B / Soul / Funk,love,347,2.98,2.11,2.11,6.92,7.11,1.63,4.90,...,14.89,0.48,5.00,0.00,2.31,0.00,0.19,9.13,0.19,0.0
9,Latin,sadness,343,1.94,0.78,4.18,10.98,4.28,3.11,5.93,...,11.56,1.26,2.04,0.00,1.94,0.00,0.87,13.41,0.49,0.0


,genre,goemotions_emotion_predominante,top3_goemotions_mas_comunes,top3_goemotions_mas_comunes_pct
0,Other,love,"approval, love, disappointment","approval (12.64%), love (11.51%), disappointme..."
1,Rock,disappointment,"approval, disappointment, sadness","approval (12.78%), disappointment (10.13%), sa..."
2,Pop,love,"love, approval, sadness","love (15.07%), approval (11.88%), sadness (9.63%)"
3,Folk / Traditional,love,"approval, love, disappointment","approval (11.71%), love (11.47%), disappointme..."
4,Electronic / Dance,love,"approval, love, sadness","approval (13.17%), love (12.6%), sadness (8.57%)"
5,Metal,annoyance,"annoyance, approval, sadness","annoyance (15.11%), approval (11.62%), sadness..."
6,Hip Hop / Rap,annoyance,"annoyance, anger, approval","annoyance (20.11%), anger (11.92%), approval (..."
7,Country,love,"love, approval, disappointment","love (13.33%), approval (11.79%), disappointme..."
8,R&B / Soul / Funk,love,"love, approval, sadness","love (14.89%), approval (13.06%), sadness (9.13%)"
9,Latin,sadness,"sadness, disappointment, love","sadness (13.41%), disappointment (12.34%), lov..."


---

### **4. Visualización resultados**



#### **4.1. Canciones por género**

Este gráfico de barras horizontales muestra el total de canciones únicas analizadas, clasificadas por género musical. Las barras se ordenan de menor a mayor cantidad, permitiendo comparar visualmente la representación de cada estilo en el dataset. Además, cada barra incluye el número exacto de pistas en su extremo derecho, facilitando la identificación rápida de los géneros musicales más y menos predominantes en la base de datos.

In [140]:
print(df_song_genre_models.shape)

print(df_genre_counts.shape)

(20291, 23)
(9189, 23)


In [141]:
# ============================================================
# 1. Number of songs per genre - versión Plotly
# ============================================================

pio.renderers.default = "vscode"

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

df_genre_counts = df_song_genre_models.copy()

df_genre_counts = df_genre_counts.drop_duplicates(
    subset=["artist_norm", "song_norm", "genre"]
)

genre_counts = (
    df_genre_counts
    .groupby("genre")
    .agg(n_songs=("song_norm", "count"))
    .reset_index()
    .sort_values("n_songs", ascending=True)
)

# ------------------------------------------------------------
# Plotly chart
# ------------------------------------------------------------

fig = px.bar(
    genre_counts,
    x="n_songs",
    y="genre",
    orientation="h",
    text="n_songs",
    title="Number of Songs per Genre",
    labels={
        "n_songs": "Number of songs",
        "genre": "Genre"
    }
)

fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    width=1000,
    height=700,
    title=dict(
        font=dict(size=20)
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray"
    ),
    yaxis=dict(
        title="Genre"
    ),
    margin=dict(l=160, r=80, t=80, b=60)
)

fig.show()



#### **4.2. Sentimiento medio por género**

Este gráfico ilustra el sentimiento promedio por género musical, ordenándolos desde los más negativos a los más positivos. Cada barra indica la cantidad de canciones analizadas (n) y cruza un eje central en cero que separa claramente las puntuaciones pesimistas de las optimistas. Además, utiliza colores para destacar la emoción predominante, permitiendo identificar al instante qué estilos musicales tienden a ser más alegres o melancólicos.

In [142]:
# ============================================================
# 2. Mean sentiment by genre
# ============================================================

pio.renderers.default = "vscode"

# ------------------------------------------------------------
# Preparar datos
# ------------------------------------------------------------

plot_data = genre_sentiment_final.copy()

plot_data["genre_label"] = (
    plot_data["genre"] + " (n=" + plot_data["n_canciones"].astype(str) + ")"
)

plot_data = plot_data.sort_values(
    "sentimiento_medio",
    ascending=True
).reset_index(drop=True)

# ------------------------------------------------------------
# Mostrar con Plotly
# ------------------------------------------------------------

color_map = {
    "negative": "#d73027",
    "neutral": "#fee08b",
    "positive": "#1a9850"
}

fig_plotly = px.bar(
    plot_data,
    x="sentimiento_medio",
    y="genre_label",
    orientation="h",
    color="sentimiento_medio_clase",
    text=plot_data["sentimiento_medio"].round(2),
    title="Sentimiento medio por género",
    labels={
        "sentimiento_medio": "Sentimiento medio",
        "genre_label": "Género",
        "sentimiento_medio_clase": "Clase según sentimiento medio"
    },
    color_discrete_map=color_map
)

fig_plotly.update_traces(
    textposition="outside"
)

# Renombrar leyenda y ocultar neutral si existe
for trace in fig_plotly.data:
    if trace.name == "negative":
        trace.name = "Negativo"
    elif trace.name == "positive":
        trace.name = "Positivo"
    elif trace.name == "neutral":
        trace.name = "Neutral"
        trace.showlegend = False

fig_plotly.add_vline(
    x=0,
    line_width=1,
    line_dash="dash",
    line_color="black"
)

fig_plotly.update_layout(
    width=1100,
    height=800,
    title=dict(font=dict(size=20)),
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(title="Género"),
    legend=dict(
        title="Clase según media"
    ),
    margin=dict(l=200, r=80, t=80, b=60)
)

fig_plotly.show()

**Sentimiento medio por género**

El gráfico muestra que la mayoría de géneros presentan un sentimiento medio positivo, aunque con diferencias claras de intensidad. Latin y Jazz aparecen como los géneros más positivos, aunque su menor número de canciones obliga a interpretarlos con cierta cautela. Entre los géneros con más representación, destacan R&B / Soul / Funk, Other, Pop, Electronic / Dance y Country, todos con una tendencia positiva moderada. Hip Hop / Rap queda prácticamente neutro, con un valor apenas positivo, mientras que Rock aparece ligeramente negativo. Los géneros con mayor carga negativa son Experimental, Metal y especialmente Punk / Emo / Hardcore, lo que sugiere una mayor presencia de letras emocionalmente intensas, críticas o asociadas a sentimientos negativos.

#### **4.3. Heatmap de sentimientos por género - Modelo J-hartmann**

Este mapa de calor visualiza la distribución porcentual de las siete emociones del modelo j-hartmann en cada género musical. Las celdas muestran el porcentaje exacto y utilizan una escala de colores cálidos donde los tonos más oscuros o intensos indican una mayor presencia emocional. Esta representación gráfica permite identificar de un solo vistazo qué emociones dominan en cada estilo y descubrir patrones clave en las letras.

In [143]:
# ============================================================
# 4. j-hartmann heatmap by genre
# ============================================================

pio.renderers.default = "vscode"

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

plot_data = genre_jhartmann_final.copy()

MIN_SONGS = 1

plot_data = plot_data[plot_data["n_canciones"] >= MIN_SONGS].copy()

plot_data = plot_data.sort_values("n_canciones", ascending=False)

plot_data["genre_label"] = (
    plot_data["genre"] + " (n=" + plot_data["n_canciones"].astype(str) + ")"
)

jhartmann_emotions = [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise"
]

jhartmann_pct_cols = [f"pct_jhartmann_{emotion}" for emotion in jhartmann_emotions]

for col in jhartmann_pct_cols:
    if col not in plot_data.columns:
        plot_data[col] = 0.0

heatmap_data = plot_data.set_index("genre_label")[jhartmann_pct_cols].copy()
heatmap_data.columns = jhartmann_emotions


# ============================================================
# Mostrar con Plotly
# ============================================================

fig_plotly = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="j-hartmann Emotion Distribution by Genre",
    labels={
        "x": "Emotion",
        "y": "Genre",
        "color": "Percentage (%)"
    }
)

fig_plotly.update_layout(
    width=1000,
    height=800,
    title=dict(font=dict(size=20)),
    margin=dict(l=220, r=80, t=80, b=80)
)

fig_plotly.update_xaxes(
    tickangle=45
)

fig_plotly.show()



#### **4.4. Emociones predominantes por género**

Este panel compuesto por múltiples subgráficos ilustra las 5 emociones predominantes del modelo GoEmotions para nueve géneros musicales específicos. Mediante barras horizontales, detalla el porcentaje exacto de cada emoción dentro de estilos como el Rock, Pop o Metal. Esta vista facilita la comparación directa, revelando rápidamente el perfil emocional complejo y los matices sentimentales únicos que caracterizan a cada género musical.

In [144]:
# ============================================================
# 5. GoEmotions por género
# ============================================================


pio.renderers.default = "vscode"

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

target_genres = [
    "Rock",
    "Electronic / Dance",
    "Pop",
    "Country",
    "Folk / Traditional",
    "R&B / Soul / Funk",
    "Metal",
    "Hip Hop / Rap",
    "Punk / Emo / Hardcore"
]

TOP_N = 7

# ------------------------------------------------------------
# Detectar columnas de GoEmotions
# ------------------------------------------------------------

goemotion_pct_cols = [
    col for col in genre_goemotions_final.columns
    if col.startswith("pct_go_")
]

# Crear dataframe largo
plot_long = genre_goemotions_final[
    ["genre"] + goemotion_pct_cols
].copy()

plot_long = plot_long[plot_long["genre"].isin(target_genres)].copy()

plot_long = plot_long.melt(
    id_vars="genre",
    value_vars=goemotion_pct_cols,
    var_name="emotion",
    value_name="percentage"
)

plot_long["emotion"] = plot_long["emotion"].str.replace("pct_go_", "", regex=False)


# ============================================================
# Mostrar con Plotly
# ============================================================

fig_plotly = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=target_genres,
    horizontal_spacing=0.12,
    vertical_spacing=0.12
)

for i, genre_name in enumerate(target_genres):
    row = i // 3 + 1
    col = i % 3 + 1

    genre_data = plot_long[plot_long["genre"] == genre_name].copy()

    genre_data = genre_data.sort_values("percentage", ascending=False).head(TOP_N)
    genre_data = genre_data.sort_values("percentage", ascending=True)

    fig_plotly.add_trace(
        go.Bar(
            x=genre_data["percentage"],
            y=genre_data["emotion"],
            orientation="h",
            text=[f"{v:.1f}%" for v in genre_data["percentage"]],
            textposition="outside",
            showlegend=False
        ),
        row=row,
        col=col
    )

    fig_plotly.update_xaxes(
        title_text="%",
        showgrid=True,
        gridcolor="lightgray",
        row=row,
        col=col
    )

    fig_plotly.update_yaxes(
        title_text="",
        row=row,
        col=col
    )

fig_plotly.update_layout(
    width=1400,
    height=1100,
    title=dict(
        text=f"Top {TOP_N} emociones GoEmotions por género",
        font=dict(size=20)
    ),
    margin=dict(l=80, r=80, t=100, b=60)
)

fig_plotly.show()


**Emociones GoEmotions por género**

El gráfico permite observar perfiles emocionales más detallados por género. 

+ En `Rock` predominan emociones negativas o introspectivas como *disappointment, sadness y annoyance*, lo que encaja con el sentimiento medio ligeramente negativo visto antes. 

+ `Electronic / Dance`, `Pop`, `Country`, `Folk / Traditional` y `R&B / Soul / Funk` tienen *love* como una de las emociones principales, aunque también aparecen con fuerza sadness y disappointment, mostrando que incluso géneros positivos mezclan emociones afectivas con melancolía. 

+ `Metal` y `Punk / Emo / Hardcore` concentran emociones como *sadness, annoyance y disappointment*, reforzando su perfil más negativo. 

+ `Hip Hop / Rap` destaca especialmente por *annoyance y anger*, diferenciándose del resto por una carga más confrontativa. 

En general, GoEmotions muestra que los géneros no se distinguen solo por positividad o negatividad, sino por combinaciones emocionales específicas.

In [147]:
# ============================================================
# Añadir columna year a df_model_comparison desde music_dataset_final.csv
# ============================================================

import pandas as pd
import re
import html

# ------------------------------------------------------------
# Cargar dataset antiguo con year
# ------------------------------------------------------------

df_year_source = pd.read_csv("music_dataset_final_masaños.csv")


# ------------------------------------------------------------
# Función de normalización por si no existen artist_norm/song_norm
# ------------------------------------------------------------

def normalize_text_id(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)

    return text

# Crear claves normalizadas en dataset con year
if "artist_norm" not in df_year_source.columns:
    df_year_source["artist_norm"] = df_year_source["artist"].apply(normalize_text_id)

if "song_norm" not in df_year_source.columns:
    df_year_source["song_norm"] = df_year_source["song"].apply(normalize_text_id)

# Crear claves normalizadas en df_model_comparison si no existen
if "artist_norm" not in df_model_comparison.columns:
    df_model_comparison["artist_norm"] = df_model_comparison["artist"].apply(normalize_text_id)

if "song_norm" not in df_model_comparison.columns:
    df_model_comparison["song_norm"] = df_model_comparison["song"].apply(normalize_text_id)

# ------------------------------------------------------------
# Limpiar year en dataset fuente
# ------------------------------------------------------------

if "year" not in df_year_source.columns:
    raise ValueError("music_dataset_final_2.csv no contiene la columna 'year'.")

df_year_source["year"] = pd.to_numeric(df_year_source["year"], errors="coerce")

df_year_source = df_year_source[
    df_year_source["year"].notna() &
    (df_year_source["year"] >= 1900) &
    (df_year_source["year"] <= 2030)
].copy()

df_year_source["year"] = df_year_source["year"].astype(int)

# ------------------------------------------------------------
# Crear lookup único artist_norm + song_norm -> year
# Si hay duplicados, usa el año más frecuente
# ------------------------------------------------------------

df_year_lookup = (
    df_year_source
    .groupby(["artist_norm", "song_norm"])["year"]
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)

print("Canciones únicas con year:", len(df_year_lookup))

# ------------------------------------------------------------
# Eliminar year anterior si existe en df_model_comparison
# ------------------------------------------------------------

if "year" in df_model_comparison.columns:
    df_model_comparison = df_model_comparison.drop(columns=["year"])

# ------------------------------------------------------------
# Añadir year a df_model_comparison
# ------------------------------------------------------------

df_model_comparison = df_model_comparison.merge(
    df_year_lookup,
    on=["artist_norm", "song_norm"],
    how="left"
)

# ------------------------------------------------------------
# Comprobación
# ------------------------------------------------------------

n_total = len(df_model_comparison)
n_with_year = df_model_comparison["year"].notna().sum()

print(f"Filas con year asignado: {n_with_year}/{n_total}")
print(f"Porcentaje con year: {round(n_with_year / n_total * 100, 2)}%")


Canciones únicas con year: 1288
Filas con year asignado: 1966/9189
Porcentaje con year: 21.4%


In [152]:
# ============================================================
# Evolución de positividad media por década - Basado en Cardiff sentiment_signed_score
# ============================================================

pio.renderers.default = "vscode"

# ------------------------------------------------------------
# 1. Preparar datos
# ------------------------------------------------------------

df_decade = df_model_comparison.copy()

# Una fila por canción
df_decade = df_decade.drop_duplicates(subset=["artist_norm", "song_norm"])

# Limpiar year
df_decade["year"] = pd.to_numeric(df_decade["year"], errors="coerce")

df_decade = df_decade[
    df_decade["year"].notna() &
    (df_decade["year"] >= 1900) &
    (df_decade["year"] <= 2030)
].copy()

df_decade["year"] = df_decade["year"].astype(int)

# Crear década
PERIOD = 10

df_decade["decade"] = (df_decade["year"] // PERIOD) * PERIOD
df_decade["decade_label"] = (
    df_decade["decade"].astype(str) + "-" + 
    (df_decade["decade"] + PERIOD - 1).astype(str)
)

# Crear score firmado si no existe
if "sentiment_signed_score" not in df_decade.columns:
    def sentiment_to_signed_score(row):
        label = row["cardiff_sentiment"]
        score = row["cardiff_sentiment_score"]

        if pd.isna(label) or pd.isna(score):
            return pd.NA

        label = str(label).lower()

        if label == "positive":
            return float(score)
        elif label == "negative":
            return -float(score)
        elif label == "neutral":
            return 0.0
        else:
            return pd.NA

    df_decade["sentiment_signed_score"] = df_decade.apply(
        sentiment_to_signed_score,
        axis=1
    )

df_decade = df_decade[df_decade["sentiment_signed_score"].notna()].copy()

print("Número de canciones válidas:", len(df_decade))


# ------------------------------------------------------------
# 2. Agregación por década
# ------------------------------------------------------------

decade_positivity = (
    df_decade
    .groupby(["decade", "decade_label"])
    .agg(
        n_songs=("song_norm", "count"),
        positividad_media=("sentiment_signed_score", "mean"),
        positividad_mediana=("sentiment_signed_score", "median"),
        desviacion_tipica=("sentiment_signed_score", "std")
    )
    .reset_index()
)

# Filtrar décadas con pocas canciones
MIN_SONGS_PER_DECADE = 10

decade_positivity = decade_positivity[
    decade_positivity["n_songs"] >= MIN_SONGS_PER_DECADE
].copy()

decade_positivity = decade_positivity.sort_values("decade").reset_index(drop=True)


# ------------------------------------------------------------
# 3. Mostrar con Plotly
# ------------------------------------------------------------

fig_plotly = px.line(
    decade_positivity,
    x="decade_label",
    y="positividad_media",
    markers=True,
    title="Evolution of Average Positivity by Decade",
    labels={
        "decade_label": "Decade",
        "positividad_media": "Average positivity score"
    },
    hover_data={
        "n_songs": True,
        "positividad_media": ":.3f",
        "positividad_mediana": ":.3f",
        "desviacion_tipica": ":.3f"
    }
)

fig_plotly.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig_plotly.update_layout(
    width=1000,
    height=650,
    title=dict(font=dict(size=20)),
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
    margin=dict(l=80, r=60, t=80, b=60)
)

fig_plotly.show()


Número de canciones válidas: 876


**Evolución de positividad media por década**

La evolución temporal indica que la positividad media de las canciones se mantiene siempre por encima de cero, por lo que el dataset presenta una tendencia general positiva en todas las décadas. 

Sin embargo, no hay una progresión lineal: la positividad es alta en los años 50, baja en los 60, vuelve a subir en los 70, cae en los 80 y alcanza su punto más bajo en la década del 2000. En los años 2020 se observa una recuperación clara. La interpretación debe considerar el número de canciones por década, ya que décadas con pocos ejemplos pueden producir valores más inestables.